In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text, inspect
from sqlalchemy import Integer, String, Float, Boolean, DateTime
import urllib.parse
import logging
import glob
import os
import sys


# ________ Logging Setup __________________________________________________________________

def setup_logging(log_file="UPI_Transactions_Pipeline.txt"):
    logger = logging.getLogger("UPITransactionPipeline")
    if logger.handlers:
        return logger

    logger.setLevel(logging.DEBUG)
    logger.propagate = False          # prevent bubbling to root logger

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    # utf-8 encoding prevents crashes on Windows when logging
    # special characters like arrows or currency symbols
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger


# ______ Stage 1 : Ingestion ________________________________________________________________

def ingest(folder_path, logger):
    logger.info("Stage 1 | Ingestion Started")
    logger.debug(f"Reading CSVs From : {folder_path}")

    if not os.path.exists(folder_path):
        logger.error(f"Folder not Found : {folder_path}")
        raise FileNotFoundError(f"Folder not Found : {folder_path}")

    dataframes = {}
    files = glob.glob(os.path.join(folder_path, "*.csv"))

    if not files:
        logger.warning("No CSV file found in folder")
        return dataframes                                           # FIX 1 : was 'dataframe' (typo)

    for file in files:
        name = os.path.basename(file).replace(".csv", "")
        try:
            dataframes[name] = pd.read_csv(file)
            logger.debug(f"Loaded '{name}'  -> {dataframes[name].shape[0]} rows , {dataframes[name].shape[1]} cols")
        except Exception as e:
            logger.error(f"Failed to read '{name}': {e}")
            raise

    logger.info(f"Stage 1 | Ingestion Completed -> {len(dataframes)} files loaded : {list(dataframes.keys())}")   # FIX 2 : was 'dtaframes' (typo) + missing closing bracket
    return dataframes


# _______ Stage 2 : Clean customer_details ____________________________________________________

def Clean_customer_details(df, logger):
    logger.info("Stage 2 | Cleaning Customer Details")
    df = df.copy()

    before = df.shape[0]
    df['date_joined'] = pd.to_datetime(df['date_joined'])
    logger.debug("Converted date_joined to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null Values after cleaning : {nulls}")
    logger.info(f"customer_details -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# _______ Stage 3 : Clean customer_feedback ____________________________________________________

def Clean_customer_feedback(df, logger):
    logger.info("Stage 3 | Cleaning customer_feedback")
    df = df.copy()

    before = df.shape[0]
    df['date_submitted'] = pd.to_datetime(df['date_submitted'])
    logger.debug("Converted date_submitted to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"customer_feedback -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# ________ Stage 4 : Clean device_info _____________________________________________________________

def Clean_device_info(df, logger):
    logger.info("Stage 4 | Cleaning device_info")
    df = df.copy()

    before = df.shape[0]
    df['last_active'] = pd.to_datetime(df['last_active'])
    logger.debug("Converted last_active to datetime")              # FIX 3 : was 'device_info' (wrong column name in message)

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"device_info -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# _________ Stage 5 : Clean fraud_alert _______________________________________________________________

def Clean_fraud_alert(df, logger):
    logger.info("Stage 5 | Cleaning fraud_alert")
    df = df.copy()

    before = df.shape[0]
    df['alert_date'] = pd.to_datetime(df['alert_date'])
    logger.debug("Converted alert_date to datetime")

    # NOTE : resolution_date nulls are expected — they belong to unresolved alerts (resolved=False)
    # These are NOT missing data, just open cases. Do not fill or drop them.

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"fraud_alert -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# __________ Stage 6 : Clean merchant __________________________________________________________________

def Clean_merchant(df, logger):
    logger.info("Stage 6 | Cleaning merchant")
    df = df.copy()

    before = df.shape[0]
    df['onboard_date'] = pd.to_datetime(df['onboard_date'])
    logger.debug("Converted onboard_date to datetime")

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"merchant -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# __________ Stage 7 : Clean account_details _____________________________________________________________

def Clean_account_details(df, logger):
    logger.info("Stage 7 | Cleaning account_details")
    df = df.copy()

    before = df.shape[0]
    df['date_added'] = pd.to_datetime(df['date_added'])
    logger.debug("Converted date_added to datetime")              # FIX 4 : was 'onboard_date' (wrong column name in message)

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"account_details -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# ____________ Stage 8 : Clean transaction_history ________________________________________________________

def Clean_transaction_history(df, logger):
    logger.info("Stage 8 | Cleaning transaction_history")         # FIX 5 : was Stage 7 (duplicate stage number)
    df = df.copy()

    before = df.shape[0]
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    logger.debug("Converted timestamp to datetime")               # FIX 6 : was 'onboard_date' (wrong column name in message)

    # NOTE : merchant_id nulls are expected — only send/receive txns have no merchant
    # NOTE : failure_reason nulls are expected — only failed txns have a reason
    # Both are structurally valid nulls, not missing data.

    nulls = df.isnull().sum().sum()
    logger.debug(f"Null values after cleaning : {nulls}")
    logger.info(f"transaction_history -> {df.shape[0]} rows ( no rows dropped from {before} )")
    return df


# ── Stage 9 : Database Engine ───────────────────────────────────────

def create_db_engine(server_name, database_name, logger):
    logger.info("Stage 9 | Creating database engine...")
    try:
        params = urllib.parse.quote_plus(
            f'DRIVER={{ODBC Driver 18 for SQL Server}};'
            f'SERVER={server_name};'
            f'DATABASE={database_name};'
            f'Trusted_Connection=yes;'
            f'Encrypt=yes;'
            f'TrustServerCertificate=yes;'
        )
        engine = create_engine(
            f"mssql+pyodbc:///?odbc_connect={params}",
            fast_executemany=True
        )
        logger.info("  Database engine created successfully.")
        return engine
    except Exception as e:
        logger.error(f"  Failed to create engine: {e}")
        raise


# ── Stage 10 : Smart Upload ──────────────────────────────────────────

def upload_to_sql(df, table_name, engine, unique_keys, dtype_map, logger):
    logger.info(f"  Uploading '{table_name}' - {len(df)} records...")

    try:
        inspector    = inspect(engine)
        table_exists = inspector.has_table(table_name)

        # CASE 1 — Fresh upload
        if not table_exists:
            logger.info(f"  Table '{table_name}' not found - fresh upload...")
            df["uploaded_at"] = pd.Timestamp.now()
            df.to_sql(
                table_name,
                engine,
                if_exists="replace",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(f"  Fresh upload complete - {len(df)} records inserted.")

        # CASE 2 — Smart insert (no duplicates)
        else:
            logger.info(f"  Table '{table_name}' exists - checking for duplicates...")

            existing_df = pd.read_sql(
                f"SELECT {', '.join(unique_keys)} FROM {table_name}",
                engine
            )
            logger.info(f"  Existing records in DB: {len(existing_df)}")

            merged   = df.merge(existing_df, on=unique_keys, how="left", indicator=True)
            new_rows = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

            if len(new_rows) == 0:                                 # FIX 7 : indentation was broken
                logger.info(f"  No new records for '{table_name}' - upload skipped.")
                return

            new_rows["uploaded_at"] = pd.Timestamp.now()
            new_rows.to_sql(
                table_name,
                engine,
                if_exists="append",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(
                f"  Smart upload complete - - "
                f"{len(new_rows)} new records inserted, "
                f"{len(df) - len(new_rows)} duplicates skipped."
            )

        # ── Row Count Validation (NEW) ──────────────────────────────
        db_count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table_name}", engine).iloc[0]['cnt']
        logger.info(f" Row count validation - DB now has {db_count} records in '{table_name}'.")

    except Exception as e:
        logger.error(f"  Upload failed for '{table_name}': {e}")
        raise


# _________ dtype_maps _____________________________________________________________________________

DTYPE_MAPS = {
    "customer_details": {
        "customer_id":      String(50),
        "full_name":        String(200),
        "mobile_number":    String(20),
        "age":              Integer(),
        "gender":           String(20),
        "region":           String(50),
        "date_joined":      DateTime(),
        "is_business_user": Boolean(),
        "risk_score":       Float()
    },
    "customer_feedback": {
        "feedback_id":        String(50),
        "customer_id":        String(50),
        "date_submitted":     DateTime(),
        "feedback_text":      String(500),
        "satisfaction_score": Integer(),
        "issue_type":         String(50),
        "resolved":           Boolean()
    },
    "device_info": {
        "device_id":   String(50),
        "customer_id": String(50),
        "device_type": String(50),
        "app_version": String(20),
        "is_rooted":   Boolean(),
        "last_active": DateTime()
    },
    "fraud_alert": {
        "alert_id":        String(50),
        "transaction_id":  String(50),
        "alert_type":      String(50),
        "alert_date":      DateTime(),
        "resolved":        Boolean(),
        "resolution_date": DateTime(),
        "remarks":         String(500)
    },
    "merchant": {
        "merchant_id":   String(50),
        "merchant_name": String(200),
        "merchant_type": String(50),
        "region":        String(50),
        "onboard_date":  DateTime(),
        "risk_score":    Float()
    },
    "account_details": {
        "upi_id":        String(50),
        "customer_id":   String(50),
        "bank_name":     String(50),
        "account_type":  String(50),
        "date_added":    DateTime(),
        "status":        String(20)
    },
    "transaction_history": {
        "transaction_id":   String(50),
        "upi_id":           String(50),
        "customer_id":      String(50),
        "timestamp":        DateTime(),
        "amount":           Float(),
        "transaction_type": String(50),
        "merchant_id":      String(50),
        "counterparty_upi": String(50),
        "status":           String(20),
        "device_id":        String(50),
        "device_type":      String(50),
        "channel":          String(50),
        "fraud_flag":       Boolean(),
        "reversal_flag":    Boolean(),
        "failure_reason":   String(100)
    }
}


# ________ Unique Keys ________________________________________________________________________

UNIQUE_KEYS = {
    "customer_details":    ["customer_id"],
    "customer_feedback":   ["feedback_id"],
    "device_info":         ["device_id"],
    "fraud_alert":         ["alert_id"],
    "merchant":            ["merchant_id"],
    "account_details":     ["upi_id"],
    "transaction_history": ["transaction_id"]
}


# ________ Master Pipeline _____________________________________________________________________

def run_pipeline(folder_path, server_name, database_name, log_file="UPI_Transactions_Pipeline.txt"):
    logger = setup_logging(log_file)
    logger.info('=' * 70)
    logger.info("UPI Transactions Pipeline - START")
    logger.info('=' * 70)

    try:
        # Cleaning Stages :
        raw                  = ingest(folder_path, logger)
        customer_details     = Clean_customer_details(raw['customer_master'], logger)          # FIX 8 : was raw['customer_master',logger) — missing closing ]
        customer_feedback    = Clean_customer_feedback(raw['customer_feedback_surveys'], logger)
        device_info          = Clean_device_info(raw['device_info'], logger)
        fraud_alert          = Clean_fraud_alert(raw['fraud_alert_history'], logger)
        merchant             = Clean_merchant(raw['merchant_info'], logger)
        account_details      = Clean_account_details(raw['upi_account_details'], logger)
        transaction_history  = Clean_transaction_history(raw['upi_transaction_history'], logger)  # FIX 9 : was raw['upi_transaction_history',logger) — missing closing ]

        # Database Stage :
        engine = create_db_engine(server_name, database_name, logger)

        logger.info("Stage 10 | Uploading all tables to SQL Server")
        upload_to_sql(customer_details,       "customer_details",       engine, UNIQUE_KEYS["customer_details"],       DTYPE_MAPS["customer_details"],       logger)
        upload_to_sql(customer_feedback,      "customer_feedback",      engine, UNIQUE_KEYS["customer_feedback"],      DTYPE_MAPS["customer_feedback"],      logger)
        upload_to_sql(device_info,            "device_info",            engine, UNIQUE_KEYS["device_info"],            DTYPE_MAPS["device_info"],            logger)
        upload_to_sql(fraud_alert,            "fraud_alert",            engine, UNIQUE_KEYS["fraud_alert"],            DTYPE_MAPS["fraud_alert"],            logger)
        upload_to_sql(merchant,               "merchant",               engine, UNIQUE_KEYS["merchant"],               DTYPE_MAPS["merchant"],               logger)
        upload_to_sql(account_details,        "account_details",        engine, UNIQUE_KEYS["account_details"],        DTYPE_MAPS["account_details"],        logger)
        upload_to_sql(transaction_history,    "transaction_history",    engine, UNIQUE_KEYS["transaction_history"],    DTYPE_MAPS["transaction_history"],    logger)

        logger.info('=' * 70)
        logger.info("UPI Transactions Pipeline - COMPLETE")
        logger.info('=' * 70)

    except Exception as e:
        logger.critical(f"Pipeline FAILED : {e}", exc_info=True)
        raise


# _________ Run ________________________________________________________________________________________

folder_path   = r'D:\\DATA SETS (Project)\\Capston Project Data\\UPI_Transactions_Data_Analysis'
server_name   = r'LAPTOP-2VUPGIQA\SQLEXPRESS'
database_name = r'UPI_Transactions_Data_DB'

run_pipeline(folder_path, server_name, database_name, log_file="UPI_Transactions_Pipeline.txt")

2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | UPI Transactions Pipeline - START
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | UPI Transactions Pipeline - START
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | ======================================================================
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Started
2026-05-26 15:34:38 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Started
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Completed -> 7 files loaded : ['customer_feedback_

C:\Users\user\AppData\Local\Temp\ipykernel_13808\2343152006.py:220: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  inspector    = inspect(engine)


2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Existing records in DB: 10000
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Existing records in DB: 10000
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   No new records for 'customer_details' - upload skipped.
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   No new records for 'customer_details' - upload skipped.
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Uploading 'customer_feedback' - 4000 records...
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Uploading 'customer_feedback' - 4000 records...
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Table 'customer_feedback' exists - checking for duplicates...
2026-05-26 15:34:39 | INFO     | UPITransactionPipeline |   Table 'customer_feedback' exists - checking for duplicates...
2026-05-26 15:34:40 | INFO     | UPITransactionPipeline |   Existing records in DB: 4000
2026-05-26 15:34:40 | INFO     | UPITra

ProgrammingError: (pyodbc.ProgrammingError) ('String data, right truncation: length 52 buffer 46', 'HY000')
[SQL: INSERT INTO fraud_alert (alert_id, transaction_id, alert_type, alert_date, resolved, resolution_date, remarks, uploaded_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [('alert183925', 'txn10083925', 'frequent_failure', datetime.datetime(2024, 10, 9, 9, 41, 15, 203880), 1, '2024-10-11 02:41:15.203880', 'Art do federal short.', datetime.datetime(2026, 5, 26, 15, 34, 40, 166510)), ('alert163439', 'txn10063439', 'unusual_time', datetime.datetime(2025, 2, 23, 1, 23, 55, 939700), 1, '2025-02-24 10:23:55.939700', 'Age board fine participant join.', datetime.datetime(2026, 5, 26, 15, 34, 40, 166510)), ('alert164967', 'txn10064967', 'frequent_failure', datetime.datetime(2025, 7, 8, 21, 50, 16, 926780), 1, '2025-07-09 19:50:16.926780', 'Among doctor much physical interest catch.', datetime.datetime(2026, 5, 26, 15, 34, 40, 166510)), ('alert159661', 'txn10059661', 'frequent_failure